In [50]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Load datasets

insurance = pd.read_csv("insurance.csv")
validation_data = pd.read_csv("validation_dataset.csv")

# Clean training dataset

def clean_training_dataset(df):
    df = df.copy()
    df['sex'] = df['sex'].replace({'M': 'male', 'man': 'male', 'F': 'female', 'woman': 'female'})
    df['charges'] = df['charges'].replace({'\$': ''}, regex=True).astype(float)
    df = df[df['age'] > 0]
    df.loc[df['children'] < 0, 'children'] = 0
    df['region'] = df['region'].str.lower()
    return df.dropna()

cleaned_insurance = clean_training_dataset(insurance)

In [51]:
# Train Linear Regression Model

def train_regression_model(df):
    X = df.drop('charges', axis=1)
    y = df['charges']

    categorical_cols = ['sex', 'smoker', 'region']
    numerical_cols = ['age', 'bmi', 'children']

    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_encoded)

    model = Pipeline([
        ('scaler', scaler),
        ('regressor', LinearRegression())
    ])

    model.fit(X_scaled, y)

    mse_scores = -cross_val_score(model, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
    r2_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')

    return model, np.mean(mse_scores), np.mean(r2_scores), X_encoded.columns

insurance_model, mean_mse, r2_score, training_columns = train_regression_model(cleaned_insurance)

print("Mean MSE:", mean_mse)
print("Mean R2:", r2_score)

Mean MSE: 37431001.52191915
Mean R2: 0.7450511466263761


In [52]:
# Clean validation dataset

def clean_validation_dataset(df):
    df = df.copy()
    df['sex'] = df['sex'].replace({'M': 'male', 'man': 'male', 'F': 'female', 'woman': 'female'})
    df.loc[df['age'] < 0, 'age'] = df['age'].abs()
    df.loc[df['children'] < 0, 'children'] = 0
    df['region'] = df['region'].str.lower()
    for col in ['age', 'bmi', 'children']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    for col in ['sex', 'smoker', 'region']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])
    return df

validation_cleaned = clean_validation_dataset(validation_data)

# Prepare validation features

categorical_cols = ['sex', 'smoker', 'region']
validation_encoded = pd.get_dummies(validation_cleaned, columns=categorical_cols, drop_first=True)

# Align columns to training data
validation_encoded = validation_encoded.reindex(columns=training_columns, fill_value=0)

In [53]:
# Predict charges and assign directly to validation_data

validation_data['predicted_charges'] = insurance_model.predict(validation_encoded)
validation_data['predicted_charges'] = np.maximum(validation_data['predicted_charges'], 1000)

# Check results

validation_data.head()

,age,sex,bmi,children,smoker,region,predicted_charges
0,18.0,female,24.090000,1.0,no,southeast,128624.195643
1,39.0,male,26.410000,0.0,yes,northeast,220740.537449
2,27.0,male,29.150000,0.0,yes,southeast,181357.588606
3,71.0,male,65.502135,13.0,yes,southeast,423490.687270
4,28.0,male,38.060000,0.0,no,southeast,193247.431989
